In [43]:
import math

import numpy as np
import pandas as pd
import plotly.express as px
import random

In [44]:
pval = 1
pi = np.array([[pval], [1-pval]])

In [45]:
#inital request's probability of being valid given data, descriptors or input
pval = 1

#component performance
p_c = .75

pi = np.array([[pval], [1-pval]])
C = np.array([[p_c, 0], [1-p_c, 1]])

In [46]:
#observed component performance
C @ pi

array([[0.75],
       [0.25]])

In [47]:
#building 

rng = np.random.default_rng(seed=42)

n_components = 6


def build_component_matrix(p_c):
    return np.array(np.array([[p_c, 0], [1-p_c, 1]]))


component_ensemble = {}

for i in range(n_components):
    component_id = f"C{i + 1}"
    p_c = rng.uniform(low=0.5, high=0.8)
    component_ensemble[component_id] = build_component_matrix(p_c)


In [48]:
rng = np.random.default_rng(seed=7)

n_users = 100


def build_human_matrix(p_ac, p_ai):
    return np.array([[p_ac, p_ai], [1 - p_ac, 1 - p_ai]])


user_ensemble = {}

for i in range(n_users):
    user_id = f"u{i + 1}"
    p_ac = 1
    p_ai = 0
    user_ensemble[user_id] = build_human_matrix(p_ac, p_ai)

In [49]:
component_ensemble.keys()

dict_keys(['C1', 'C2', 'C3', 'C4', 'C5', 'C6'])

In [50]:
component_ensemble.keys()

dict_keys(['C1', 'C2', 'C3', 'C4', 'C5', 'C6'])

In [51]:
import numpy as np

def link_system_components(ensemble, component_list):
    result = np.eye(2)
    for key in component_list:
        result = np.matmul(result, ensemble[key])
        
    return result

In [52]:
#improvement rate measures as (1-c11)*r per execusion (cadence controlled in the simulation step)
def component_improvement_sprint(component_ensemble, component_list, improvement_rate):
    for key in component_list:
        improvement = (1-component_ensemble[key][0][0])*improvement_rate
        component_ensemble[key][0][0] += improvement
        component_ensemble[key][1][0] -= improvement
    return component_ensemble

In [53]:
def simulate_daily_activity(day, n_items, user_ensemble, component_ensemble):
    pi = np.array([[1],[0]])
    telemetry = []
    user_ids = list(user_ensemble.keys())
    component_ids = list(component_ensemble.keys())

    for _ in range(n_items):
        user = rng.choice(user_ids)
        system_inference_path = random.sample(component_ids, random.randint(2,6))
        composite_system_performance = link_system_components(component_ensemble, system_inference_path)
        accept_prob = float((user_ensemble[user] @ composite_system_performance @ pi)[0, 0])
        is_accepted = rng.random() < accept_prob
        telemetry.append(
            [day, user, is_accepted, system_inference_path]
        )

    return pd.DataFrame(
        telemetry,
        columns=[
            "Date",
            "User",
            "isAccepted",
            "systemInferencePath",
        ],
    )

In [54]:
simulate_daily_activity(1,1000,user_ensemble, component_ensemble)

,Date,User,isAccepted,systemInferencePath
0,1,u95,False,"[C2, C4, C5, C6, C1]"
1,1,u63,False,"[C1, C6, C2, C3, C5]"
2,1,u84,False,"[C3, C1, C6, C5, C2]"
3,1,u23,False,"[C5, C6, C3, C2, C4, C1]"
4,1,u92,False,"[C6, C4, C3, C5]"
...,...,...,...,...
995,1,u40,True,"[C5, C1]"
996,1,u16,False,"[C6, C4, C1, C5, C2]"
997,1,u43,True,"[C4, C6, C1, C2]"
998,1,u100,False,"[C3, C6, C5, C1, C2]"


In [56]:
n_items_per_day = 1000
simulation_duration = 365 + 90

historical_telemetry = []

for day_idx in range(simulation_duration):
    # system_correct_prob = min(system_quality_over_time(day_idx), 1.0)
    historical_telemetry.append(
        simulate_daily_activity(
            day=day_idx + 1,
            n_items=n_items_per_day,
            user_ensemble=user_ensemble,
            component_ensemble=component_ensemble,
        )
    )

    if (day_idx + 1) % 7 == 0:
        if day_idx < 365 + 60:
            component_ensemble = component_improvement_sprint(component_ensemble, random.sample(list(component_ensemble.keys()), 1), .4)

historical_telemetry_df = pd.concat(historical_telemetry, ignore_index=True)
historical_telemetry_df.head()

,Date,User,isAccepted,systemInferencePath
0,1,u54,False,"[C2, C6, C1]"
1,1,u65,True,"[C5, C4, C3]"
2,1,u12,False,"[C2, C1, C6, C5]"
3,1,u19,False,"[C5, C2, C1, C3, C4, C6]"
4,1,u89,False,"[C5, C3, C6, C4]"


In [57]:
historical_telemetry_df

,Date,User,isAccepted,systemInferencePath
0,1,u54,False,"[C2, C6, C1]"
1,1,u65,True,"[C5, C4, C3]"
2,1,u12,False,"[C2, C1, C6, C5]"
3,1,u19,False,"[C5, C2, C1, C3, C4, C6]"
4,1,u89,False,"[C5, C3, C6, C4]"
...,...,...,...,...
454995,455,u72,True,"[C4, C5]"
454996,455,u44,True,"[C5, C1, C3, C2]"
454997,455,u55,True,"[C2, C4, C6, C3, C1, C5]"
454998,455,u9,False,"[C4, C1, C2, C6, C3, C5]"


In [58]:
observed_daily_accepts = (
    historical_telemetry_df.groupby(["Date"])["isAccepted"]
    .mean()
    .reset_index()
)

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted"
)
fig.update_traces(marker={"size": 5})
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [59]:
historical_telemetry_df.systemInferencePath.apply(lambda x: x.sort())

0         None
1         None
2         None
3         None
4         None
          ... 
454995    None
454996    None
454997    None
454998    None
454999    None
Name: systemInferencePath, Length: 455000, dtype: object

In [60]:
historical_telemetry_df.systemInferencePath

0                     [C1, C2, C6]
1                     [C3, C4, C5]
2                 [C1, C2, C5, C6]
3         [C1, C2, C3, C4, C5, C6]
4                 [C3, C4, C5, C6]
                    ...           
454995                    [C4, C5]
454996            [C1, C2, C3, C5]
454997    [C1, C2, C3, C4, C5, C6]
454998    [C1, C2, C3, C4, C5, C6]
454999    [C1, C2, C3, C4, C5, C6]
Name: systemInferencePath, Length: 455000, dtype: object

In [61]:
expanded_df = historical_telemetry_df.systemInferencePath.apply(pd.Series)

# Optional: Rename the new columns (e.g., C_1, C_2...)
expanded_df = expanded_df.rename(columns=lambda x: f'Component_{x+1}')

In [62]:
expanded_df = ~expanded_df.isna()

In [63]:
list(expanded_df.columns)

['Component_1',
 'Component_2',
 'Component_3',
 'Component_4',
 'Component_5',
 'Component_6']

In [64]:
historical_telemetry_df[list(expanded_df.columns)] = expanded_df

In [65]:
categories = {}
cc = 1
for inf_path in historical_telemetry_df.systemInferencePath.astype('str').unique():
    categories[inf_path] = "Type_" + str(cc)
    cc += 1

In [66]:
historical_telemetry_df["request_type"] = historical_telemetry_df.systemInferencePath.apply(lambda x: categories[str(x)])

In [67]:
historical_telemetry_df

,Date,User,isAccepted,systemInferencePath,Component_1,Component_2,Component_3,Component_4,Component_5,Component_6,request_type
0,1,u54,False,"[C1, C2, C6]",True,True,True,False,False,False,Type_1
1,1,u65,True,"[C3, C4, C5]",True,True,True,False,False,False,Type_2
2,1,u12,False,"[C1, C2, C5, C6]",True,True,True,True,False,False,Type_3
3,1,u19,False,"[C1, C2, C3, C4, C5, C6]",True,True,True,True,True,True,Type_4
4,1,u89,False,"[C3, C4, C5, C6]",True,True,True,True,False,False,Type_5
...,...,...,...,...,...,...,...,...,...,...,...
454995,455,u72,True,"[C4, C5]",True,True,False,False,False,False,Type_52
454996,455,u44,True,"[C1, C2, C3, C5]",True,True,True,True,False,False,Type_54
454997,455,u55,True,"[C1, C2, C3, C4, C5, C6]",True,True,True,True,True,True,Type_4
454998,455,u9,False,"[C1, C2, C3, C4, C5, C6]",True,True,True,True,True,True,Type_4


In [68]:
daily_accept_stats = (
    historical_telemetry_df.groupby(["Date", "request_type"], as_index=False)
    .agg(
        accepted_sum=("isAccepted", "sum"),
        n_obs=("isAccepted", "size"),
    )
)

if pd.api.types.is_datetime64_any_dtype(daily_accept_stats["Date"]):
    all_dates = pd.date_range(
        daily_accept_stats["Date"].min(),
        daily_accept_stats["Date"].max(),
        freq="D",
        name="Date",
    )
else:
    all_dates = pd.Index(
        range(daily_accept_stats["Date"].min(), daily_accept_stats["Date"].max() + 1),
        name="Date",
    )

observed_daily_accepts = (
    daily_accept_stats.set_index(["request_type", "Date"])
    .reindex(
        pd.MultiIndex.from_product(
            [daily_accept_stats["request_type"].unique(), all_dates],
            names=["request_type", "Date"],
        ),
        fill_value=0,
    )
    .reset_index()
    .sort_values(["request_type", "Date"])
)

observed_daily_accepts[["accepted_sum_7d", "n_obs_7d"]] = (
    observed_daily_accepts.groupby("request_type")[["accepted_sum", "n_obs"]]
    .transform(lambda s: s.rolling(window=7, min_periods=1).sum())
)

observed_daily_accepts["isAccepted"] = (
    observed_daily_accepts["accepted_sum_7d"] / observed_daily_accepts["n_obs_7d"]
)

observed_daily_accepts = observed_daily_accepts[observed_daily_accepts["n_obs"] > 0]

fig = px.scatter(
    observed_daily_accepts,
    x="Date",
    y="isAccepted",
    color="request_type",
    hover_name="request_type",
    opacity=0.45,
    title="7-Day Trailing Acceptance Rate by Request Type",
    labels={"isAccepted": "Acceptance Rate", "Date": "Day"},
)
fig.update_traces(marker={"size": 5})
# fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [69]:
select_telemetry = historical_telemetry_df.loc[historical_telemetry_df.Date >= 365 + 60]

In [70]:
select_telemetry.columns

Index(['Date', 'User', 'isAccepted', 'systemInferencePath', 'Component_1',
       'Component_2', 'Component_3', 'Component_4', 'Component_5',
       'Component_6', 'request_type'],
      dtype='object')

In [71]:
select_telemetry[['Date','User']]

,Date,User
424000,425,u9
424001,425,u90
424002,425,u49
424003,425,u96
424004,425,u3
...,...,...
454995,455,u72
454996,455,u44
454997,455,u55
454998,455,u9


In [77]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.5000000	best: 0.5000000 (0)	total: 6.55ms	remaining: 13.1s
200:	test: 0.4854405	best: 0.5028073 (10)	total: 1.43s	remaining: 12.8s
400:	test: 0.4864411	best: 0.5028073 (10)	total: 3s	remaining: 12s
600:	test: 0.4864411	best: 0.5028073 (10)	total: 4.63s	remaining: 10.8s
800:	test: 0.4864411	best: 0.5028073 (10)	total: 6.12s	remaining: 9.16s
1000:	test: 0.4864411	best: 0.5028073 (10)	total: 7.53s	remaining: 7.51s
1200:	test: 0.4864411	best: 0.5028073 (10)	total: 9.01s	remaining: 5.99s
1400:	test: 0.4864411	best: 0.5028073 (10)	total: 10.5s	remaining: 4.48s
1600:	test: 0.4864411	best: 0.5028073 (10)	total: 11.9s	remaining: 2.96s
1800:	test: 0.4864411	best: 0.5028073 (10)	total: 13.2s	remaining: 1.46s
1999:	test: 0.4864411	best: 0.5028073 (10)	total: 14.6s	remaining: 0us

bestTest = 0.5028072747
bestIteration = 10

Shrink model to first 11 iterations.


In [78]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",  # good for your 7.5% positive rate
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,          # <--- names, not indices
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.4925543	best: 0.4925543 (0)	total: 5.96ms	remaining: 11.9s
200:	test: 0.4966924	best: 0.5078303 (118)	total: 1.57s	remaining: 14s
400:	test: 0.4921726	best: 0.5149382 (329)	total: 3.25s	remaining: 13s
600:	test: 0.4666254	best: 0.5149382 (329)	total: 5.04s	remaining: 11.7s
800:	test: 0.4585753	best: 0.5149382 (329)	total: 6.86s	remaining: 10.3s
1000:	test: 0.4722931	best: 0.5149382 (329)	total: 8.72s	remaining: 8.7s
1200:	test: 0.4920148	best: 0.5149382 (329)	total: 10.6s	remaining: 7.04s
1400:	test: 0.4981051	best: 0.5149382 (329)	total: 12.3s	remaining: 5.27s
1600:	test: 0.5019919	best: 0.5149382 (329)	total: 14s	remaining: 3.5s
1800:	test: 0.5073177	best: 0.5149382 (329)	total: 15.8s	remaining: 1.74s
1999:	test: 0.5086457	best: 0.5149382 (329)	total: 17.6s	remaining: 0us

bestTest = 0.5149382318
bestIteration = 329

Shrink model to first 330 iterations.


In [79]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
y = df["isAccepted"].astype(int)
X = df[['Date','User', 'request_type']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()


model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.5443588	best: 0.5443588 (0)	total: 6.99ms	remaining: 14s
200:	test: 0.6077478	best: 0.6253618 (5)	total: 1.69s	remaining: 15.1s
400:	test: 0.5748116	best: 0.6253618 (5)	total: 3.68s	remaining: 14.7s
600:	test: 0.5659940	best: 0.6253618 (5)	total: 5.72s	remaining: 13.3s
800:	test: 0.5537047	best: 0.6253618 (5)	total: 7.71s	remaining: 11.5s
1000:	test: 0.5556978	best: 0.6253618 (5)	total: 9.77s	remaining: 9.75s
1200:	test: 0.5652833	best: 0.6253618 (5)	total: 12s	remaining: 7.98s
1400:	test: 0.5632914	best: 0.6253618 (5)	total: 14.1s	remaining: 6.02s
1600:	test: 0.5603865	best: 0.6253618 (5)	total: 16.2s	remaining: 4.04s
1800:	test: 0.5633697	best: 0.6253618 (5)	total: 18.3s	remaining: 2.02s
1999:	test: 0.5600440	best: 0.6253618 (5)	total: 20.3s	remaining: 0us

bestTest = 0.6253617953
bestIteration = 5

Shrink model to first 6 iterations.


In [81]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
# df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']] = df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']].astype(int)
y = df["isAccepted"].astype(int)
X = df[['Date', 'User', 'Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6', 'request_type']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.5930469	best: 0.5930469 (0)	total: 11.8ms	remaining: 23.6s
200:	test: 0.5927359	best: 0.6267324 (9)	total: 1.78s	remaining: 15.9s
400:	test: 0.5756883	best: 0.6267324 (9)	total: 3.98s	remaining: 15.9s
600:	test: 0.5748174	best: 0.6267324 (9)	total: 6.33s	remaining: 14.7s
800:	test: 0.5608219	best: 0.6267324 (9)	total: 8.73s	remaining: 13.1s
1000:	test: 0.5520991	best: 0.6267324 (9)	total: 11.2s	remaining: 11.2s
1200:	test: 0.5652757	best: 0.6267324 (9)	total: 13.5s	remaining: 9.01s
1400:	test: 0.5713286	best: 0.6267324 (9)	total: 15.8s	remaining: 6.75s
1600:	test: 0.5648829	best: 0.6267324 (9)	total: 17.9s	remaining: 4.47s
1800:	test: 0.5639267	best: 0.6267324 (9)	total: 20.2s	remaining: 2.23s
1999:	test: 0.5637292	best: 0.6267324 (9)	total: 22.3s	remaining: 0us

bestTest = 0.6267324094
bestIteration = 9

Shrink model to first 10 iterations.


In [84]:
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd

df = select_telemetry.copy()
df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']] = df[['Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6']].astype(int)
y = df["isAccepted"].astype(int)
X = df[['Date', 'User', 'Component_1','Component_2', 'Component_3', 'Component_4', 'Component_5','Component_6', 'request_type']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    auto_class_weights="Balanced",
    depth=6,
    learning_rate=0.05,
    iterations=2000,
    # random_seed=42,
    verbose=200
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.6004914	best: 0.6004914 (0)	total: 18.4ms	remaining: 36.8s
200:	test: 0.5961020	best: 0.6207239 (22)	total: 1.88s	remaining: 16.9s
400:	test: 0.5914103	best: 0.6207239 (22)	total: 3.93s	remaining: 15.7s
600:	test: 0.5941901	best: 0.6207239 (22)	total: 6.06s	remaining: 14.1s
800:	test: 0.5860085	best: 0.6207239 (22)	total: 8.21s	remaining: 12.3s
1000:	test: 0.5769958	best: 0.6207239 (22)	total: 10.3s	remaining: 10.3s
1200:	test: 0.5701971	best: 0.6207239 (22)	total: 12.5s	remaining: 8.28s
1400:	test: 0.5698452	best: 0.6207239 (22)	total: 14.8s	remaining: 6.31s
1600:	test: 0.5658883	best: 0.6207239 (22)	total: 17s	remaining: 4.24s
1800:	test: 0.5606080	best: 0.6207239 (22)	total: 19.3s	remaining: 2.13s
1999:	test: 0.5572613	best: 0.6207239 (22)	total: 21.6s	remaining: 0us

bestTest = 0.620723918
bestIteration = 22

Shrink model to first 23 iterations.


In [ ]:
#Seems like model path is not enough for errors to be separable even with perfect humans